# LSI Lite – lastoptimiert slicen aus einem FEM-Ergebnis

Aus einer `.vtu`-Datei mit Spannungsergebnis (z. B. FreeCAD/CalculiX) entsteht G-Code für einen
3-Achs-Drucker, dessen Schichten der Zugspannung folgen.

**So geht's:** Zellen der Reihe nach mit ▶ starten (oder *Laufzeit → Alle ausführen*).
1. **Einrichten** – einmal je Sitzung, etwa 20 Sekunden
2. **Datei hochladen** – deine `.vtu`
3. **Einstellungen und Rechnen** – Formular rechts ausfüllen, dann starten
4. **Ansehen** – Bahnen in 3D, mit Regler „bis Schicht"
5. **Herunterladen** – den G-Code

Vereinfachte Fassung des LSI aus der Diplomarbeit von Gereon Zündorf; Grenzen siehe README.
Den G-Code vor dem Druck immer ansehen.

In [ ]:
#@title 1  Einrichten
import os, subprocess, sys
VERSION = "main"   # fester Stand, z. B. "v1.0"
URL = f"https://raw.githubusercontent.com/gereon1112-web/lsi-lite/{VERSION}/lsi_lite.py"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "meshio==5.3.5"], check=True)
if not os.path.exists("lsi_lite.py"):
    subprocess.run(["wget", "-q", "-O", "lsi_lite.py", URL], check=True)
import lsi_lite
print("Bereit.")

In [ ]:
#@title 2  Datei hochladen
from google.colab import files
hochgeladen = files.upload()
vtu = next((name for name in hochgeladen if name.lower().endswith(".vtu")), None)
print(f"Eingabe: {vtu}" if vtu else "Keine .vtu-Datei dabei - bitte erneut hochladen.")

In [ ]:
#@title 3  Einstellungen und Rechnen
duese = 0.4  #@param {type:"number"}
schicht_min = 0.1  #@param {type:"number"}
schicht_max = 0.35  #@param {type:"number"}
neigung_max = 20  #@param {type:"slider", min:5, max:45, step:1}
fuellung = "konzentrisch"  #@param ["konzentrisch", "zigzag"]
fuelldichte = 100  #@param {type:"slider", min:0, max:100, step:5}
waende = 1  #@param {type:"slider", min:0, max:5, step:1}
drucker = "220 x 220 (Ender-3 u. a.)"  #@param ["220 x 220 (Ender-3 u. a.)", "250 x 210 (Prusa MK3/MK4)", "256 x 256 (Bambu Lab X1/P1)", "350 x 350 (Voron 2.4 u. a.)"]
material = "PLA"  #@param ["PLA", "PETG", "ABS"]
baurichtung = "+z"  #@param ["+z", "-z", "+x", "-x", "+y", "-y"]
#@markdown *Baurichtung*: welche Achse des FEM-Modells beim Druck nach oben zeigt.

bahnen = None
if not vtu:
    print("Erst in Zelle 2 eine .vtu-Datei hochladen.")
else:
    try:
        gcode, bild, bahnen = lsi_lite.slice_vtu(
            vtu, nozzle=duese, layer_min=schicht_min, layer_max=schicht_max,
            max_tilt=neigung_max, infill=fuellung, density=fuelldichte, walls=waende,
            printer=drucker, material=material, build_dir=baurichtung)
    except lsi_lite.InputError as fehler:
        print("Abbruch:", fehler)

In [ ]:
#@title 4  Ansehen: Bahnen in 3D (drehen mit der Maus)
bis_schicht = 0  #@param {type:"integer"}
#@markdown 0 = alle Schichten. Kleinere Zahl zeigt den Stand mitten im Druck, z. B. die Mittelschicht.
if bahnen is None:
    print("Erst in Zelle 3 rechnen.")
else:
    lsi_lite.show_paths(bahnen, upto=bis_schicht or None).show()

In [ ]:
#@title 5  G-Code herunterladen
files.download(str(gcode))